## **SNAP Jupyter demo notebook**
**RCM Compact Polarimetry — CP-RVI, Stokes parameters, m–χ / m–δ / H–α decompositions**

In summary, this workflow contains:

- Background on compact polarimetry (CP) and the RCM CP modes
- Part 1: CP Radar Vegetation Index (`Compactpol-Radar-Vegetation-Index`)
- Part 2: Stokes parameters (`CP-Stokes-Parameters`)
- Part 3: CP decompositions — m–χ (RGB), m–δ, H–α (`CP-Decomposition`)

Complexity: advanced

##### ***About the test data:***

RADARSAT Constellation Mission (RCM) Compact Polarimetric SLC products are **not** bundled here. Free RCM data is available through the [NRCan EO Data Discovery Portal](https://eodms-sgdot.nrcan-rncan.gc.ca/) (registration required).

Look for product filenames containing `*16MCP*` (16-bit compact-pol) or `*32MCP*` (float compact-pol). Either CH/CV or RH/RV transmit polarisations work with the operators below.

Place the unzipped product directory under `data/` and update the path.

##### ***Some information on the Python environment:***

In [ ]:
import os
import sys
print("Python version: " + sys.version)

import sysconfig
print("Location of esa_snappy package: " + sysconfig.get_paths()['purelib'] + os.sep + "esa_snappy")

##### ***Import Python packages...***

In [ ]:
import esa_snappy
from esa_snappy import ProductIO

import snapista
from snapista import Graph
from snapista import Operator

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

##### ***Convenience plot functions:***

In [ ]:
def _read_band(product, name):
    band = product.getBand(name)
    if band is None:
        raise KeyError(f"Band {name!r} not found. Available: {[b.getName() for b in product.getBands()]}")
    w, h = band.getRasterWidth(), band.getRasterHeight()
    data = np.zeros(w * h, np.float32)
    band.readPixels(0, 0, w, h, data)
    data.shape = h, w
    return data

def plot_band(product, name, title=None, cmap='viridis', vmin=None, vmax=None):
    data = _read_band(product, name)
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title or name)
    fig.colorbar(im, ax=ax)
    plt.show()

def plot_interferogram(product, phase_band_name, coh_band_name=None):
    phi = _read_band(product, phase_band_name)
    if coh_band_name is None:
        fig, ax = plt.subplots(figsize=(7, 6))
        im = ax.imshow(phi, cmap='hsv', vmin=-np.pi, vmax=np.pi)
        ax.set_title('Wrapped interferogram phase [rad]')
        fig.colorbar(im, ax=ax)
    else:
        coh = _read_band(product, coh_band_name)
        fig, axs = plt.subplots(1, 2, figsize=(13, 5))
        im0 = axs[0].imshow(phi, cmap='hsv', vmin=-np.pi, vmax=np.pi)
        axs[0].set_title('Wrapped phase [rad]')
        fig.colorbar(im0, ax=axs[0])
        im1 = axs[1].imshow(coh, cmap='viridis', vmin=0, vmax=1)
        axs[1].set_title('Coherence')
        fig.colorbar(im1, ax=axs[1])
    plt.show()

def find_band(product, *patterns):
    names = [b.getName() for b in product.getBands()]
    for pat in patterns:
        for n in names:
            if pat.lower() in n.lower():
                return n
    raise KeyError(f"No band matching {patterns!r} found. Available: {names}")

---

### ***Background: compact polarimetry***

A **fully polarimetric** (quad-pol) SAR transmits H and V alternately and receives both H and V on each pulse — four scattering matrix elements per pixel. It's the gold standard but costs power and halves the swath width.

**Compact polarimetry** is a compromise: transmit one *circular* (or 45° linear) polarisation, receive both H and V coherently. You get *two* complex channels (RH+RV or LH+LV) carrying enough of the polarimetric scattering information for many vegetation, soil-moisture and ship-detection applications, but with full swath and quasi-linear power budget.

The dominant CP information is captured in the 2×2 **covariance matrix C₂** (equivalently the **Stokes vector g**). The notebook uses three SNAP operators that consume these forms:

| Operator | What it computes |
|:---------|:-----------------|
| `Compactpol-Radar-Vegetation-Index` | CP-RVI — a single scalar vegetation index analogous to NDVI |
| `CP-Stokes-Parameters` | Stokes vector + derived parameters (degree of polarisation, ellipticity, conformity, χ, αₛ, ϕ) |
| `CP-Decomposition` | One of: **m–χ** (RGB), **m–δ**, **H–α C2** — separate scattering mechanisms |

RCM's CP mode transmits Right Circular (RH/RV); the SNAP operators auto-detect the compact-pol mode from the product metadata.

---

### ***Configure input paths***

In [ ]:
data_dir = os.path.join(os.getcwd(), 'data')
graphs_dir = os.path.join(os.getcwd(), 'graphs')
results_dir = os.path.join(os.getcwd(), 'results')
os.makedirs(graphs_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

# --- cached data fetch (public S3 / HTTP; re-runs reuse the local copy) ---
import urllib.request as _urlreq, zipfile as _zip, glob as _glob

def fetch_cached(url, dest_dir):
    """Download `url` into dest_dir unless already present, unzip a .zip, and return the path to
    open (manifest.safe for a .SAFE product, else the downloaded file). Cached for re-runs:
    if the product is already in dest_dir it is NOT downloaded again."""
    os.makedirs(dest_dir, exist_ok=True)
    fname = url.split('/')[-1]
    stem = fname[:-4] if fname.lower().endswith('.zip') else fname
    hits = _glob.glob(os.path.join(dest_dir, stem + '*', 'manifest.safe'))
    if hits:
        print('cached:', os.path.basename(os.path.dirname(hits[0]))); return hits[0]
    local = os.path.join(dest_dir, fname)
    if not os.path.exists(local):
        print('downloading', fname, '...')
        _urlreq.urlretrieve(url, local)
        print('  saved %.0f MB' % (os.path.getsize(local) / 1e6))
    if fname.lower().endswith('.zip'):
        with _zip.ZipFile(local) as z:
            z.extractall(dest_dir)
        hits = _glob.glob(os.path.join(dest_dir, stem + '*', 'manifest.safe'))
        return hits[0] if hits else local
    return local

rcm_cp_product = os.path.join(data_dir, 'RCM1_OK<id>_PK<id>_<mode>MCP<n>_<date>', 'product.xml')
# RCM CP products use product.xml as their manifest; if your reader prefers manifest.safe, switch accordingly.
# To auto-download inputs from a public bucket, use fetch_cached, e.g.:
#   product = fetch_cached('https://<bucket>/<product>.zip', data_dir)


---
## ***Part 1 — CP Radar Vegetation Index***
---

Simplest possible CP product: a single-band vegetation index derived from the C₂ matrix invariants. The chain is `Read → Calibration → Compactpol-Radar-Vegetation-Index → Terrain-Correction → Write`.

In [ ]:
g_rvi = Graph()
g_rvi.add_node(operator=Operator('Read', file=rcm_cp_product), node_id='Read')
g_rvi.add_node(operator=Operator('Calibration',
                                 outputImageInComplex='true',
                                 outputSigmaBand='false'),
               node_id='Calib', source='Read')
g_rvi.add_node(operator=Operator('Compactpol-Radar-Vegetation-Index'),
               node_id='CPRVI', source='Calib')
g_rvi.add_node(operator=Operator('Terrain-Correction',
                                 demName='Copernicus 30m Global DEM',
                                 mapProjection='WGS84(DD)',
                                 pixelSpacingInMeter='10',
                                 imgResamplingMethod='BILINEAR_INTERPOLATION'),
               node_id='TC', source='CPRVI')

rvi_out = os.path.join(results_dir, 'snap_nb_rcm_cprvi.dim')
g_rvi.add_node(operator=Operator('Write', file=rvi_out, formatName='BEAM-DIMAP'),
               node_id='Write', source='TC')

g_rvi.save_graph(os.path.join(graphs_dir, 'snap_nb_rcm_cprvi.xml'))
g_rvi.run()

In [ ]:
p_rvi = ProductIO.readProduct(rvi_out)
print('Bands:', [b.getName() for b in p_rvi.getBands()])
rvi = find_band(p_rvi, 'CPRVI', 'RVI')
plot_band(p_rvi, rvi, title='CP-RVI', cmap='YlGn', vmin=0, vmax=1)
p_rvi.dispose()

---
## ***Part 2 — Stokes parameters***
---

The Stokes vector `(g₀, g₁, g₂, g₃)` fully characterises the polarisation state of the received wave. The `CP-Stokes-Parameters` operator outputs the Stokes vector itself plus derived quantities:

- **Degree of polarisation (m)** — 0 to 1, how polarised vs depolarised the return is
- **Ellipticity χ** — sign indicates dominant scattering type (left/right circular)
- **Conformity μ** — ratio used to distinguish surface vs volume scattering
- **αₛ** (alpha-s) — the compact-pol analogue of full-pol alpha angle
- **Phase ϕ** — phase difference between H and V receive channels

In [ ]:
g_stokes = Graph()
g_stokes.add_node(operator=Operator('Read', file=rcm_cp_product), node_id='Read')
g_stokes.add_node(operator=Operator('Calibration',
                                    outputImageInComplex='true',
                                    outputSigmaBand='false'),
                  node_id='Calib', source='Read')
g_stokes.add_node(operator=Operator('CP-Stokes-Parameters',
                                    windowSizeXStr='5',
                                    windowSizeYStr='5',
                                    outputStokesVector='true',
                                    outputDegreeOfPolarization='true',
                                    outputDegreeOfEllipticity='true',
                                    outputAlphas='true'),
                  node_id='Stokes', source='Calib')
g_stokes.add_node(operator=Operator('Terrain-Correction',
                                    demName='Copernicus 30m Global DEM',
                                    mapProjection='WGS84(DD)',
                                    pixelSpacingInMeter='10',
                                    imgResamplingMethod='BILINEAR_INTERPOLATION'),
                  node_id='TC', source='Stokes')

stokes_out = os.path.join(results_dir, 'snap_nb_rcm_stokes.dim')
g_stokes.add_node(operator=Operator('Write', file=stokes_out, formatName='BEAM-DIMAP'),
                  node_id='Write', source='TC')

g_stokes.save_graph(os.path.join(graphs_dir, 'snap_nb_rcm_stokes.xml'))
g_stokes.run()

In [ ]:
p_stokes = ProductIO.readProduct(stokes_out)
print('Bands:', [b.getName() for b in p_stokes.getBands()])

dop = find_band(p_stokes, 'DegreeOfPolarization', 'DegreeOfPol')
ellipticity = find_band(p_stokes, 'DegreeOfEllipticity', 'Ellipticity', 'Chi')
alphas = find_band(p_stokes, 'Alphas', 'AlphaS', 'Alpha')

fig, axs = plt.subplots(1, 3, figsize=(16, 5))
for ax, name, title, cmap in [
    (axs[0], dop, 'Degree of Polarisation (m)', 'viridis'),
    (axs[1], ellipticity, 'Ellipticity χ [rad]', 'RdBu_r'),
    (axs[2], alphas, 'αₛ [rad]', 'plasma'),
]:
    data = _read_band(p_stokes, name)
    im = ax.imshow(data, cmap=cmap)
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
plt.show()
p_stokes.dispose()

---
## ***Part 3 — CP decompositions***
---

We'll run three different decompositions on the same scene, each producing its characteristic three-component output:

- **m–χ** — splits return into double-bounce, volume, and surface based on the degree of polarisation `m` and ellipticity `χ`. Displayed as RGB.
- **m–δ** — same idea but using `δ` (relative phase). Slightly different sensitivity to oriented targets.
- **H–α C2** — dual-pol Cloude-Pottier eigen-decomposition: outputs Entropy and mean Alpha (no Anisotropy in the dual-pol case).

In [ ]:
def cp_decomp_chain(decomposition_name, out_file):
    g = Graph()
    g.add_node(operator=Operator('Read', file=rcm_cp_product), node_id='Read')
    g.add_node(operator=Operator('Calibration',
                                 outputImageInComplex='true',
                                 outputSigmaBand='false'),
               node_id='Calib', source='Read')
    g.add_node(operator=Operator('CP-Decomposition',
                                 decomposition=decomposition_name,
                                 windowSizeXStr='5',
                                 windowSizeYStr='5'),
               node_id='Decomp', source='Calib')
    g.add_node(operator=Operator('Terrain-Correction',
                                 demName='Copernicus 30m Global DEM',
                                 mapProjection='WGS84(DD)',
                                 pixelSpacingInMeter='10',
                                 imgResamplingMethod='BILINEAR_INTERPOLATION'),
               node_id='TC', source='Decomp')
    g.add_node(operator=Operator('Write', file=out_file, formatName='BEAM-DIMAP'),
               node_id='Write', source='TC')
    g.run()

mchi_out = os.path.join(results_dir, 'snap_nb_rcm_mchi.dim')
mdelta_out = os.path.join(results_dir, 'snap_nb_rcm_mdelta.dim')
halpha_out = os.path.join(results_dir, 'snap_nb_rcm_halpha.dim')

cp_decomp_chain('M-Chi Decomposition', mchi_out); print('m-chi done')
cp_decomp_chain('M-Delta Decomposition', mdelta_out); print('m-delta done')
cp_decomp_chain('H-Alpha Decomposition', halpha_out); print('H-alpha done')

##### ***Display m–χ as RGB:***

In [ ]:
p_mchi = ProductIO.readProduct(mchi_out)
print('Bands:', [b.getName() for b in p_mchi.getBands()])

r = find_band(p_mchi, 'm_chi_db', 'm-chi_db', 'm_chi_r')
g = find_band(p_mchi, 'm_chi_g')
b = find_band(p_mchi, 'm_chi_b', 'm_chi_s')

rgb = np.stack([_read_band(p_mchi, r),
                _read_band(p_mchi, g),
                _read_band(p_mchi, b)], axis=-1)
p98 = np.percentile(rgb, 98)
if p98 > 0:
    rgb = np.clip(rgb / p98, 0, 1)
fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(rgb)
ax.set_title('m–χ decomposition: double-bounce (R) / volume (G) / surface (B)')
plt.show()
p_mchi.dispose()

##### ***Display m–δ components:***

In [ ]:
p_mdelta = ProductIO.readProduct(mdelta_out)
print('Bands:', [b.getName() for b in p_mdelta.getBands()])

# Plot the three m-delta components individually
fig, axs = plt.subplots(1, 3, figsize=(16, 5))
names = [b.getName() for b in p_mdelta.getBands() if 'm_delta' in b.getName().lower() or 'm-delta' in b.getName().lower()][:3]
for ax, name in zip(axs, names):
    data = _read_band(p_mdelta, name)
    p98 = np.nanpercentile(data, 98)
    im = ax.imshow(data, cmap='magma', vmin=0, vmax=p98 if p98 > 0 else None)
    ax.set_title(name)
    fig.colorbar(im, ax=ax)
plt.show()
p_mdelta.dispose()

##### ***Display H–α (Entropy + Alpha):***

In [ ]:
p_halpha = ProductIO.readProduct(halpha_out)
print('Bands:', [b.getName() for b in p_halpha.getBands()])

H = find_band(p_halpha, 'Entropy')
alpha = find_band(p_halpha, 'Alpha')
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for ax, name, title, cmap in [
    (axs[0], H, 'Entropy (H)', 'viridis'),
    (axs[1], alpha, 'mean Alpha (°)', 'plasma'),
]:
    data = _read_band(p_halpha, name)
    im = ax.imshow(data, cmap=cmap)
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
plt.show()
p_halpha.dispose()

---

### ***Summary***

What have we learnt in this notebook?

- **Compact polarimetry** is the practical middle ground between dual-pol and full quad-pol: full swath, single transmit polarisation, dual coherent receive.
- RCM's CP mode is supported by three SNAP operators: `Compactpol-Radar-Vegetation-Index` for a vegetation index, `CP-Stokes-Parameters` for Stokes-derived quantities, and `CP-Decomposition` for component decompositions.
- **CP-RVI** — single-band quick-look for vegetation density.
- **Stokes parameters** — the lowest-level descriptors; everything else is derived from them.
- **m–χ / m–δ** — model-based three-component decompositions, good for visual scene classification.
- **H–α Dual Pol** — the dual-pol cousin of the quad-pol Cloude-Pottier eigen-decomposition; outputs Entropy and Alpha (no Anisotropy).